# VSC to TIF

Given one `.vsc` file, convert it to `.raw` (plus `_ascan_info.txt`), then to `.tif`, then save the envelope (amplitude) volume as `_amplitude.tif`.

All outputs are written next to the source `.vsc` file. Existing outputs are kept unless the overwrite switches are set.

In [ ]:
from pathlib import Path
import sys

sys.path.append(r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\10_code\UTvsXCT-preprocessing')
sys.path.append(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/UTvsXCT-preprocessing')

from matplotlib import pyplot as plt

from preprocess_tools import io, signal
from preprocess_tools.vsc_to_raw import vsc_to_raw

## 1. User inputs

In [ ]:
VSC_PATH = Path(r'/path/to/file.vsc')

RAW_DTYPE = "uint8"
BYTE_ORDER = "little"
USE_MEMMAP = True

OVERWRITE_RAW = False
OVERWRITE_TIF = False

## 2. Convert VSC to RAW

In [ ]:
vsc_path = Path(VSC_PATH)
existing = sorted(vsc_path.parent.glob(f'{vsc_path.stem}_ascan*.raw'))

if existing and not OVERWRITE_RAW:
    raw_path = existing[0]
    print(f'RAW already exists, keeping existing file: {raw_path}')
else:
    result = vsc_to_raw(vsc_path)
    raw_path = result['raw']
    X, Y, Z = result['shape']
    print(f'Saved RAW: {raw_path}  ({X}x{Y}x{Z})')

## 3. Convert RAW to TIF

In [ ]:
tif_path = raw_path.with_suffix('.tif')

if tif_path.exists() and not OVERWRITE_TIF:
    print(f'TIF already exists, keeping existing file: {tif_path}')
else:
    saved_path = io.raw_to_tif(
        raw_path,
        tif_path,
        dtype=RAW_DTYPE,
        byte_order=BYTE_ORDER,
        use_memmap=USE_MEMMAP,
    )
    print(f'Saved TIF: {saved_path}')

created_volume = io.load_tif(tif_path)
raw_shape = io.infer_raw_shape(raw_path)

plt.imshow(created_volume.max(axis=0))
plt.title('Maximum Intensity Projection')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

print(f'Inferred RAW shape (z, y, x): {raw_shape}')
print(f'Loaded TIF shape: {created_volume.shape}')
print(f'Loaded TIF dtype: {created_volume.dtype}')

assert tuple(created_volume.shape) == tuple(raw_shape), f'Shape mismatch: {created_volume.shape} != {raw_shape}'

## 4. Save amplitude volume

Envelope of the RF volume along the time axis (axis 0).

In [ ]:
amplitude_tif_path = tif_path.with_stem(tif_path.stem + '_amplitude')

amplitude_volume = signal.envelope(created_volume, axis=0)
io.save_tif(amplitude_tif_path, amplitude_volume)

plt.imshow(amplitude_volume.max(axis=0))
plt.title('Maximum Intensity Projection (amplitude)')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

print(f'Saved amplitude TIF: {amplitude_tif_path}')
print(f'Amplitude volume shape: {amplitude_volume.shape}')
print(f'Amplitude volume dtype: {amplitude_volume.dtype}')